# Blood Atlas preprocessing QC

Read-only QC for the normalized-expression-only Blood Atlas workflow.

This dataset is intentionally different from the other benchmark datasets:

- genuine raw gene-level counts are unavailable;
- the contributor-provided expression matrix is already transformed;
- expression is **not normalized/log-transformed again**;
- scVI is **not run**;
- benchmark representations are HVG, PCA, and Harmony only.

In [4]:
from pathlib import Path
import json

import anndata as ad
import numpy as np
import pandas as pd
import yaml
from scipy import sparse

def find_repo_root(
    start=Path.cwd(),
):
    start = Path(
        start
    ).resolve()

    for p in [
        start,
        *start.parents,
    ]:
        if (
            p
            / "src"
            / "scrna_benchmark"
        ).is_dir():
            return p

    raise RuntimeError(
        "Run this notebook "
        "from inside the repository."
    )

def sample_values(
    matrix,
    n=100_000,
):
    vals = (
        np.asarray(
            matrix.data
        )
        if sparse.issparse(
            matrix
        )
        else np.asarray(
            matrix
        ).ravel()
    )

    return vals[
        : min(
            n,
            len(vals),
        )
    ]

REPO_ROOT = find_repo_root()

STAGE0_PATH = (
    REPO_ROOT
    / "data"
    / "blood_atlas"
    / "blood_atlas_subsampled.h5ad"
)

ADATA_PATH = (
    REPO_ROOT
    / "data"
    / "blood_atlas"
    / "blood_atlas_benchmark_ready.h5ad"
)

QC_DIR = (
    REPO_ROOT
    / "results"
    / "preprocessing"
    / "blood_atlas"
)

print(REPO_ROOT)

/users/xchen5/scRNA-cross-donor-generalization


## Stage-0 checkpoint

In [5]:
if not STAGE0_PATH.exists():
    raise FileNotFoundError(
        STAGE0_PATH
    )

stage0 = ad.read_h5ad(
    STAGE0_PATH,
    backed="r",
)

stage0_summary = {
    "cells":
        stage0.n_obs,
    "genes":
        stage0.n_vars,
    "donors":
        stage0.obs[
            "donor_id"
        ]
        .astype(str)
        .nunique(),
    "cell_types":
        stage0.obs[
            "cell_type"
        ]
        .astype(str)
        .nunique(),
    "batches":
        stage0.obs[
            "batch"
        ]
        .astype(str)
        .nunique(),
    "has_counts":
        "counts"
        in stage0.layers,
    "has_scvi":
        "X_scVI"
        in stage0.obsm,
}

display(
    pd.DataFrame(
        [stage0_summary]
    )
)

assert stage0_summary == {
    "cells": 108682,
    "genes": 36601,
    "donors": 166,
    "cell_types": 7,
    "batches": 14,
    "has_counts": False,
    "has_scvi": False,
}

# Materialize only a small slice of the backed matrix.
# Do not pass the backed _CSRDataset directly to NumPy.
n_cells_check = min(
    1000,
    stage0.n_obs,
)
n_genes_check = min(
    1000,
    stage0.n_vars,
)

vals = stage0.X[
    :n_cells_check,
    :n_genes_check,
]

if sparse.issparse(vals):
    vals = vals.data
else:
    vals = np.asarray(
        vals
    ).ravel()

vals = vals[
    np.isfinite(vals)
]

fraction_noninteger = np.mean(
    ~np.isclose(
        vals,
        np.round(vals),
        atol=1e-6,
    )
)

print(
    f"Sampled stored values: "
    f"{vals.size:,}"
)
print(
    f"Fraction non-integer-like: "
    f"{fraction_noninteger:.3%}"
)

assert fraction_noninteger > 0

print(
    "Confirmed: Stage-0 X is "
    "transformed expression, not counts."
)

,cells,genes,donors,cell_types,batches,has_counts,has_scvi
0,108682,36601,166,7,14,False,False


Sampled stored values: 44,396
Fraction non-integer-like: 99.998%
Confirmed: Stage-0 X is transformed expression, not counts.


## Final benchmark-ready object

In [6]:
if not ADATA_PATH.exists():
    raise FileNotFoundError(
        ADATA_PATH
    )

adata = ad.read_h5ad(
    ADATA_PATH,
    backed="r",
)

observed = {
    "cells":
        adata.n_obs,
    "HVGs":
        adata.n_vars,
    "donors":
        adata.obs[
            "donor_id"
        ]
        .astype(str)
        .nunique(),
    "cell_types":
        adata.obs[
            "cell_type"
        ]
        .astype(str)
        .nunique(),
    "batches":
        adata.obs[
            "batch"
        ]
        .astype(str)
        .nunique(),
    "PCA_dims":
        adata.obsm[
            "X_pca"
        ].shape[1],
    "Harmony_dims":
        adata.obsm[
            "X_harmony"
        ].shape[1],
    "has_scVI":
        "X_scVI"
        in adata.obsm,
    "has_counts":
        "counts"
        in adata.layers,
}

display(
    pd.DataFrame(
        [observed]
    )
)

assert observed == {
    "cells": 108682,
    "HVGs": 1000,
    "donors": 166,
    "cell_types": 7,
    "batches": 14,
    "PCA_dims": 15,
    "Harmony_dims": 15,
    "has_scVI": False,
    "has_counts": False,
}

print(
    "Final object has only "
    "HVG/PCA/Harmony representations, "
    "as intended."
)

,cells,HVGs,donors,cell_types,batches,PCA_dims,Harmony_dims,has_scVI,has_counts
0,108682,1000,166,7,14,15,15,False,False


Final object has only HVG/PCA/Harmony representations, as intended.


## Verify the no-reprocessing configuration

In [7]:
with (
    QC_DIR
    / "resolved_config.yaml"
).open() as f:
    resolved = yaml.safe_load(
        f
    )

assert (
    resolved[
        "counts"
    ][
        "source"
    ]
    == "X"
)

assert (
    resolved[
        "counts"
    ][
        "require_integer_like"
    ]
    is False
)

assert (
    resolved[
        "counts"
    ][
        "prepare_from_counts"
    ]
    is False
)

assert (
    resolved[
        "scvi"
    ][
        "enabled"
    ]
    is False
)

assert (
    set(
        resolved[
            "representations"
        ]
    )
    == {
        "hvg",
        "pca",
        "harmony",
    }
)

assert (
    resolved[
        "downsample"
    ][
        "enabled"
    ]
    is False
)

print(
    "Configuration confirms: "
    "no re-normalization and no scVI."
)

Configuration confirms: no re-normalization and no scVI.


## Stage summaries

In [8]:
with (
    QC_DIR
    / "preprocessing_summary.json"
).open() as f:
    summary = json.load(
        f
    )

display(
    pd.DataFrame(
        summary[
            "stages"
        ]
    ).T
)

display(
    pd.DataFrame(
        summary[
            "expected_checks"
        ]
    ).T
)

assert (
    summary[
        "expression_provenance"
    ][
        "prepare_from_counts"
    ]
    is False
)

assert (
    summary[
        "expression_provenance"
    ][
        "scvi_enabled"
    ]
    is False
)

assert (
    set(
        summary[
            "representations"
        ]
    )
    == {
        "hvg",
        "pca",
        "harmony",
    }
)

# Stage 0 already froze the 108,682-cell population.
for stage in [
    "after_celltype_support_filter",
    "after_downsampling",
    "benchmark_ready",
]:
    assert (
        summary[
            "stages"
        ][
            stage
        ][
            "n_cells"
        ]
        == 108682
    )

,n_cells,n_genes,n_donors,n_celltypes
loaded,108682,36601,166,7
after_dataset_filters,108682,36601,166,7
after_celltype_support_filter,108682,36601,166,7
after_downsampling,108682,36601,166,7
after_gene_filtering,108682,36601,166,7
benchmark_ready,108682,1000,166,7


,expected,observed,ok
n_cells,108682,108682,True
n_donors,166,166,True
n_celltypes,7,7,True
n_hvg,1000,1000,True


## Cell-type, donor, and batch composition

In [9]:
support_before = pd.read_csv(
    QC_DIR
    / "celltype_support_before_filter.csv"
)

support_final = pd.read_csv(
    QC_DIR
    / "celltype_support_final.csv"
)

celltype_counts = pd.read_csv(
    QC_DIR
    / "celltype_counts_final.csv"
)

donor_counts = pd.read_csv(
    QC_DIR
    / "donor_counts_final.csv"
)

display(
    support_before
)
display(
    support_final
)
display(
    celltype_counts
)
display(
    donor_counts.describe()
)

batch_counts = (
    adata.obs[
        "batch"
    ]
    .astype(str)
    .value_counts()
    .rename_axis(
        "batch"
    )
    .rename(
        "n_cells"
    )
    .reset_index()
)

display(
    batch_counts
)

,cell_type,n_cells,n_donors,keep_by_cell_count,keep_by_donor_coverage,keep
0,Myeloid cells,16600,166,True,True,True
1,CD4+ T cells,16600,166,True,True,True
2,TRAV1-2- CD8+ T cells,16575,166,True,True,True
3,NK cells,16464,166,True,True,True
4,B cells,16297,166,True,True,True
5,gd T cells,14085,166,True,True,True
6,MAIT cells,12061,166,True,True,True


,cell_type,n_cells,n_donors,keep_by_cell_count,keep_by_donor_coverage,keep
0,Myeloid cells,16600,166,True,True,True
1,CD4+ T cells,16600,166,True,True,True
2,TRAV1-2- CD8+ T cells,16575,166,True,True,True
3,NK cells,16464,166,True,True,True
4,B cells,16297,166,True,True,True
5,gd T cells,14085,166,True,True,True
6,MAIT cells,12061,166,True,True,True


,cell_type,n_cells
0,Myeloid cells,16600
1,CD4+ T cells,16600
2,TRAV1-2- CD8+ T cells,16575
3,NK cells,16464
4,B cells,16297
5,gd T cells,14085
6,MAIT cells,12061


,n_cells
count,166.000000
mean,654.710843
std,54.067529
min,400.000000
25%,629.250000
50%,669.000000
75%,700.000000
max,700.000000


,batch,n_cells
0,AS061,10806
1,AS059,9935
2,AS055,9005
3,AS056,8802
4,AS058,8497
5,AS060,8293
6,AS054,7874
7,AS051,7685
8,AS049,7511
9,AS053,7410


## Provenance files

In [10]:
for name in [
    "selected_cells.csv",
    "hvg_genes.csv",
    "software_versions.json",
    "resolved_config.yaml",
]:
    path = QC_DIR / name

    print(
        name,
        (
            "OK"
            if path.exists()
            else "MISSING"
        ),
    )

    assert path.exists()

selected = pd.read_csv(
    QC_DIR
    / "selected_cells.csv"
)

hvgs = pd.read_csv(
    QC_DIR
    / "hvg_genes.csv"
)

assert len(selected) == 108682
assert len(hvgs) == 1000

if (
    hasattr(
        stage0,
        "file",
    )
    and stage0.file
    is not None
):
    stage0.file.close()

if (
    hasattr(
        adata,
        "file",
    )
    and adata.file
    is not None
):
    adata.file.close()

print(
    "Blood Atlas normalized-expression-only "
    "QC passed."
)

selected_cells.csv OK
hvg_genes.csv OK
software_versions.json OK
resolved_config.yaml OK
Blood Atlas normalized-expression-only QC passed.
